In [1]:
import pandas as pd
import numpy as np

In [2]:
features_df = pd.read_csv('../data/features_processed.csv')
features_df.head()

,date,home_team,away_team,home_GF5,home_GA5,home_W5,home_D5,home_L5,home_points5,home_mean_competition5,...,home_unbeaten_streak,away_unbeaten_streak,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches,neutral,competition_level,home_score,away_score
0,2000-01-04,Egypt,Togo,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,2,1
1,2000-01-07,Tunisia,Togo,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,7,0
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,0,0
3,2000-01-09,Mexico,Iran,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,1,1,2,1
4,2000-01-09,Ivory Coast,Egypt,0.0,0.0,0,0,0,0,0.0,...,0,1,0,0,0,0,0,1,2,0


In [3]:
features_df["date"] = pd.to_datetime(features_df["date"])

In [4]:
selected_features_m4 = [
    # Últimos 10 partidos
    "home_GF10",
    "home_GA10",
    "home_points10",
    "home_mean_competition10",

    "away_GF10",
    "away_GA10",
    "away_points10",
    "away_mean_competition10",

    # Elo
    "home_elo",
    "away_elo",
    "elo_diff",

    # H2H
    "h2h_home_wins",
    "h2h_away_wins",
    "h2h_draws",
    "h2h_matches",

    # Contexto
    "neutral",
    "competition_level"
]

X_m4 = features_df[selected_features_m4]

y_home = features_df["home_score"]
y_away = features_df["away_score"]

In [5]:
features_df["date"] = pd.to_datetime(features_df["date"])

train_mask = features_df["date"] < "2023-01-01"

val_mask = (
    (features_df["date"] >= "2023-01-01") &
    (features_df["date"] < "2025-01-01")
)

test_mask = features_df["date"] >= "2025-01-01"

X_train = X_m4[train_mask]
X_val = X_m4[val_mask]
X_test = X_m4[test_mask]

y_home_train = y_home[train_mask]
y_home_val = y_home[val_mask]
y_home_test = y_home[test_mask]

y_away_train = y_away[train_mask]
y_away_val = y_away[val_mask]
y_away_test = y_away[test_mask]

In [6]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, accuracy_score

In [7]:


param_grid = {
    "n_estimators": [200, 300, 500, 700, 1000],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7, 10],
    "gamma": [0, 0.05, 0.1, 0.2, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [0.5, 1, 2, 5]
}

In [8]:
home_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

home_search = RandomizedSearchCV(
    estimator=home_base,
    param_distributions=param_grid,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

home_search.fit(X_train, y_home_train)

print("Mejores parámetros HOME:")
print(home_search.best_params_)
print("Mejor MAE CV HOME:", home_search.best_score_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Mejores parámetros HOME:
{'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.01, 'n_estimators': 700, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 1.0}
Mejor MAE CV HOME: -1.0806142489115398


In [9]:
away_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

away_search = RandomizedSearchCV(
    estimator=away_base,
    param_distributions=param_grid,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

away_search.fit(X_train, y_away_train)

print("Mejores parámetros AWAY:")
print(away_search.best_params_)
print("Mejor MAE CV AWAY:", -away_search.best_score_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Mejores parámetros AWAY:
{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.03, 'gamma': 0.05, 'colsample_bytree': 0.6}
Mejor MAE CV AWAY: 0.8723044196764628


In [10]:
home_model_opt = home_search.best_estimator_
away_model_opt = away_search.best_estimator_

home_pred_val = home_model_opt.predict(X_val)
away_pred_val = away_model_opt.predict(X_val)

mae_home = mean_absolute_error(y_home_val, home_pred_val)
mae_away = mean_absolute_error(y_away_val, away_pred_val)

rmse_home = root_mean_squared_error(y_home_val, home_pred_val)
rmse_away = root_mean_squared_error(y_away_val, away_pred_val)

print("MAE Home:", mae_home)
print("MAE Away:", mae_away)
print("RMSE Home:", rmse_home)
print("RMSE Away:", rmse_away)

MAE Home: 1.03447687625885
MAE Away: 0.8506901860237122
RMSE Home: 1.3799042701721191
RMSE Away: 1.1127609014511108


In [11]:
def get_outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

home_pred_round = np.round(home_pred_val).astype(int)
away_pred_round = np.round(away_pred_val).astype(int)

home_pred_round = np.clip(home_pred_round, 0, None)
away_pred_round = np.clip(away_pred_round, 0, None)

results_opt = pd.DataFrame({
    "home_real": y_home_val.values,
    "away_real": y_away_val.values,
    "home_pred": home_pred_round,
    "away_pred": away_pred_round
})

results_opt["real_outcome"] = results_opt.apply(
    lambda row: get_outcome(row["home_real"], row["away_real"]),
    axis=1
)

results_opt["pred_outcome"] = results_opt.apply(
    lambda row: get_outcome(row["home_pred"], row["away_pred"]),
    axis=1
)

accuracy_wdl = accuracy_score(
    results_opt["real_outcome"],
    results_opt["pred_outcome"]
)

exact_score = (
    (results_opt["home_real"] == results_opt["home_pred"]) &
    (results_opt["away_real"] == results_opt["away_pred"])
).mean()

print("Accuracy W-D-L:", accuracy_wdl)
print("Exact Score Accuracy:", exact_score)

Accuracy W-D-L: 0.5601750547045952
Exact Score Accuracy: 0.10940919037199125


## Probando el test

In [12]:
home_pred_test = home_model_opt.predict(X_test)
away_pred_test = away_model_opt.predict(X_test)

In [13]:
mae_home_test = mean_absolute_error(y_home_test, home_pred_test)
mae_away_test = mean_absolute_error(y_away_test, away_pred_test)

rmse_home_test = root_mean_squared_error(y_home_test, home_pred_test)
rmse_away_test = root_mean_squared_error(y_away_test, away_pred_test)

print("MAE Home:", mae_home_test)
print("MAE Away:", mae_away_test)

print("RMSE Home:", rmse_home_test)
print("RMSE Away:", rmse_away_test)

MAE Home: 1.0571590662002563
MAE Away: 0.868817150592804
RMSE Home: 1.3834443092346191
RMSE Away: 1.1773724555969238


In [14]:
home_pred_round_test = np.round(home_pred_test).astype(int)
away_pred_round_test = np.round(away_pred_test).astype(int)

home_pred_round_test = np.clip(home_pred_round_test, 0, None)
away_pred_round_test = np.clip(away_pred_round_test, 0, None)

In [15]:
results_test = pd.DataFrame({
    "home_real": y_home_test.values,
    "away_real": y_away_test.values,
    "home_pred": home_pred_round_test,
    "away_pred": away_pred_round_test
})

In [16]:
def get_outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

In [17]:
results_test["real_outcome"] = results_test.apply(
    lambda row: get_outcome(row["home_real"], row["away_real"]),
    axis=1
)

results_test["pred_outcome"] = results_test.apply(
    lambda row: get_outcome(row["home_pred"], row["away_pred"]),
    axis=1
)

In [18]:

accuracy_wdl_test = accuracy_score(
    results_test["real_outcome"],
    results_test["pred_outcome"]
)

print("Accuracy WDL Test:", accuracy_wdl_test)

Accuracy WDL Test: 0.5766641735228123


In [19]:
exact_score_test = (
    (results_test["home_real"] == results_test["home_pred"])
    &
    (results_test["away_real"] == results_test["away_pred"])
).mean()

print("Exact Score Test:", exact_score_test)

Exact Score Test: 0.09798055347793568


In [20]:
test_metrics = pd.DataFrame({
    "Metric": [
        "MAE Home",
        "MAE Away",
        "RMSE Home",
        "RMSE Away",
        "Accuracy WDL",
        "Exact Score"
    ],
    "Value": [
        mae_home_test,
        mae_away_test,
        rmse_home_test,
        rmse_away_test,
        accuracy_wdl_test,
        exact_score_test
    ]
})

print(test_metrics)

         Metric     Value
0      MAE Home  1.057159
1      MAE Away  0.868817
2     RMSE Home  1.383444
3     RMSE Away  1.177372
4  Accuracy WDL  0.576664
5   Exact Score  0.097981


## Probando Mexico vs Sudafrica

In [21]:
selected_features_m4 = [
    "home_GF10",
    "home_GA10",
    "home_points10",
    "home_mean_competition10",

    "away_GF10",
    "away_GA10",
    "away_points10",
    "away_mean_competition10",

    "home_elo",
    "away_elo",
    "elo_diff",

    "h2h_home_wins",
    "h2h_away_wins",
    "h2h_draws",
    "h2h_matches",

    "neutral",
    "competition_level"
]

In [22]:
def get_latest_team_features(features_df, team, match_date):
    match_date = pd.to_datetime(match_date)

    team_matches = features_df[
        (
            (features_df["home_team"] == team) |
            (features_df["away_team"] == team)
        ) &
        (features_df["date"] < match_date)
    ].sort_values("date")

    if len(team_matches) == 0:
        raise ValueError(f"No hay partidos previos para {team}")

    last_match = team_matches.iloc[-1]

    if last_match["home_team"] == team:
        return {
            "GF10": last_match["home_GF10"],
            "GA10": last_match["home_GA10"],
            "points10": last_match["home_points10"],
            "mean_competition10": last_match["home_mean_competition10"],
            "elo": last_match["home_elo"]
        }
    else:
        return {
            "GF10": last_match["away_GF10"],
            "GA10": last_match["away_GA10"],
            "points10": last_match["away_points10"],
            "mean_competition10": last_match["away_mean_competition10"],
            "elo": last_match["away_elo"]
        }

In [23]:
def get_h2h_features(features_df, home_team, away_team, match_date):
    match_date = pd.to_datetime(match_date)

    h2h_matches = features_df[
        (
            (
                (features_df["home_team"] == home_team) &
                (features_df["away_team"] == away_team)
            )
            |
            (
                (features_df["home_team"] == away_team) &
                (features_df["away_team"] == home_team)
            )
        )
        &
        (features_df["date"] < match_date)
    ]

    h2h_home_wins = 0
    h2h_away_wins = 0
    h2h_draws = 0

    for _, row in h2h_matches.iterrows():
        if row["home_score"] == row["away_score"]:
            h2h_draws += 1

        elif row["home_score"] > row["away_score"]:
            winner = row["home_team"]

            if winner == home_team:
                h2h_home_wins += 1
            else:
                h2h_away_wins += 1

        else:
            winner = row["away_team"]

            if winner == home_team:
                h2h_home_wins += 1
            else:
                h2h_away_wins += 1

    return {
        "h2h_home_wins": h2h_home_wins,
        "h2h_away_wins": h2h_away_wins,
        "h2h_draws": h2h_draws,
        "h2h_matches": len(h2h_matches)
    }

In [24]:
def build_future_match_features(
    features_df,
    home_team,
    away_team,
    match_date,
    competition_level,
    neutral
):
    features_df = features_df.copy()
    features_df["date"] = pd.to_datetime(features_df["date"])

    home_f = get_latest_team_features(features_df, home_team, match_date)
    away_f = get_latest_team_features(features_df, away_team, match_date)

    h2h_f = get_h2h_features(features_df, home_team, away_team, match_date)

    row = {
        "home_GF10": home_f["GF10"],
        "home_GA10": home_f["GA10"],
        "home_points10": home_f["points10"],
        "home_mean_competition10": home_f["mean_competition10"],

        "away_GF10": away_f["GF10"],
        "away_GA10": away_f["GA10"],
        "away_points10": away_f["points10"],
        "away_mean_competition10": away_f["mean_competition10"],

        "home_elo": home_f["elo"],
        "away_elo": away_f["elo"],
        "elo_diff": home_f["elo"] - away_f["elo"],

        "h2h_home_wins": h2h_f["h2h_home_wins"],
        "h2h_away_wins": h2h_f["h2h_away_wins"],
        "h2h_draws": h2h_f["h2h_draws"],
        "h2h_matches": h2h_f["h2h_matches"],

        "neutral": neutral,
        "competition_level": competition_level
    }

    match_features = pd.DataFrame([row])

    return match_features[selected_features_m4]

In [25]:
def predict_match_score(
    features_df,
    home_team,
    away_team,
    match_date,
    competition_level,
    neutral,
    home_model,
    away_model
):
    match_features = build_future_match_features(
        features_df=features_df,
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        competition_level=competition_level,
        neutral=neutral
    )

    home_pred_raw = home_model.predict(match_features)[0]
    away_pred_raw = away_model.predict(match_features)[0]

    home_pred = int(round(home_pred_raw))
    away_pred = int(round(away_pred_raw))

    home_pred = max(home_pred, 0)
    away_pred = max(away_pred, 0)

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_pred_raw": home_pred_raw,
        "away_pred_raw": away_pred_raw,
        "home_pred": home_pred,
        "away_pred": away_pred,
        "match_features": match_features
    }

In [26]:
prediction = predict_match_score(
    features_df=features_df,
    home_team="United States",
    away_team="Paraguay",
    match_date="2026-06-12",
    competition_level=6,
    neutral=0,
    home_model=home_model_opt,
    away_model=away_model_opt
)

print("Predicción cruda:")
print(prediction["home_team"], prediction["home_pred_raw"])
print(prediction["away_team"], prediction["away_pred_raw"])

print("\nMarcador predicho:")
print(
    f'{prediction["home_team"]} {prediction["home_pred"]} - '
    f'{prediction["away_pred"]} {prediction["away_team"]}'
)

Predicción cruda:
United States 1.8204405
Paraguay 0.8025194

Marcador predicho:
United States 2 - 1 Paraguay


In [27]:
prediction = predict_match_score(
    features_df=features_df,
    home_team="United States",
    away_team="Paraguay",
    match_date="2026-06-12",
    competition_level=6,
    neutral=0,
    home_model=home_model_opt,
    away_model=away_model_opt
)

print("Predicción cruda:")
print(prediction["home_team"], prediction["home_pred_raw"])
print(prediction["away_team"], prediction["away_pred_raw"])

print("\nMarcador predicho:")
print(
    f'{prediction["home_team"]} {prediction["home_pred"]} - '
    f'{prediction["away_pred"]} {prediction["away_team"]}'
)

Predicción cruda:
United States 1.8204405
Paraguay 0.8025194

Marcador predicho:
United States 2 - 1 Paraguay


In [53]:
import joblib

joblib.dump(home_model_opt, "models/home_model.pkl")
joblib.dump(away_model_opt, "models/away_model.pkl")

['models/away_model.pkl']

In [30]:
features_df.to_csv("../data/features_final.csv", index=False)

In [34]:
X_home_final = features_df[selected_features_m4]
y_home_final = features_df["home_score"]


X_away_final = features_df[selected_features_m4]
y_away_final = features_df["away_score"]

In [38]:
best_params_home = home_model_opt.get_params()
best_params_away = away_model_opt.get_params()

final_home_model = XGBRegressor(
    **best_params_home,
)

final_away_model = XGBRegressor(
    **best_params_away,
)

final_home_model.fit(X_home_final, y_home_final)
final_away_model.fit(X_away_final, y_away_final)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.6
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [41]:
import joblib

In [42]:
joblib.dump(final_home_model, "../models/home_model.pkl")
joblib.dump(final_away_model, "../models/away_model.pkl")

['../models/away_model.pkl']